## Scraper completo de ccnorte.com

Une en un solo sitio las dos fases que antes estaban repartidas entre `CCNORTE.ipynb` y `ccNorte_v2.ipynb`:

1. **Calendario** (`crawl_all`): descarga el listado de carreras de ccnorte.com/resultados -> `ccnorte_resultados.csv` (nombre_carrera, fecha, circuito_nombre, categoria, categoria_url...).
2. **Clasificaciones** (`crawl_clasificaciones`, versión con el método del máximo por prefijo — arregla el sobrecomptaje de versiones anteriores): visita cada categoría y cuenta participantes por género, arrastrando `fecha`/`circuito_nombre` desde el paso 1 -> `ccnorte_participantes_resumen.csv` ya con TODO lo necesario, sin que haga falta ningún join después.

Al final hay una función `scrape_ccnorte_completo()` que encadena las dos fases con una sola llamada.

**Esto solo funciona ejecutado desde tu ordenador con Jupyter (no desde el sandbox de Claude, que no tiene acceso a internet a este dominio).**

### Fase 1 — Calendario (`crawl_all`)

In [82]:
from __future__ import annotations

import csv
import time
from pathlib import Path
from typing import Optional
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://ccnorte.com"
LIST_URL_TEMPLATE = BASE_URL + "/resultados/{page}"
FIRST_PAGE_URL = BASE_URL + "/resultados"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/126.0 Safari/537.36"
    ),
    "Accept-Language": "es-ES,es;q=0.9,ca;q=0.8,gl;q=0.7",
}

CSV_FIELDS = [
    "pagina",
    "nombre_carrera",
    "fecha",
    "circuito_nombre",
    "circuito_url",
    "imagen_url",
    "categoria",
    "categoria_url",
    "categoria_es_interna",
]


def _clean(text: Optional[str]) -> Optional[str]:
    if text is None:
        return None
    text = " ".join(text.split())
    return text or None


def _es_enlace_interno_resultados(url: str) -> bool:
    return url.startswith("/resultados/") or url.startswith(BASE_URL + "/resultados/")


def parse_resultados_page(html: str, base_url: str = BASE_URL) -> list[dict]:
    """Parsea una página del listado y devuelve una lista de dicts (una fila por categoría)."""
    soup = BeautifulSoup(html, "html.parser")
    rows: list[dict] = []

    for article in soup.select("div.resultados-page-new > article.row.section-margin"):
        h2 = article.select_one("h2.h2-titulo-resultados")
        nombre = _clean(h2.get_text()) if h2 else None
        if not nombre:
            continue

        fecha_span = article.select_one("span.span-fecha-resultados")
        fecha = _clean(fecha_span.get_text()) if fecha_span else None

        img = article.select_one(".grid-item-resultado-fixed img")
        imagen_url = urljoin(base_url, img["src"]) if img and img.get("src") else None

        circuito_a = article.select_one("div.clearfix a.label") or article.select_one("h4 a")
        circuito_nombre = _clean(circuito_a.get_text()) if circuito_a else None
        circuito_url = (
            urljoin(base_url, circuito_a["href"]) if circuito_a and circuito_a.get("href") else None
        )

        base_row = {
            "nombre_carrera": nombre,
            "fecha": fecha,
            "circuito_nombre": circuito_nombre,
            "circuito_url": circuito_url,
            "imagen_url": imagen_url,
        }

        lis = article.select(
            "div.div-header-resultados ul.resultados-carreras li.resultados-carreras__item"
        )
        any_cat = False
        for li in lis:
            a = li.find("a")
            if not a:
                continue
            cat_nombre = _clean(a.get_text())
            href = a.get("href")
            if not cat_nombre or not href:
                continue
            any_cat = True
            rows.append(
                {
                    **base_row,
                    "categoria": cat_nombre,
                    "categoria_url": urljoin(base_url, href),
                    "categoria_es_interna": _es_enlace_interno_resultados(href),
                }
            )
        if not any_cat:
            rows.append(
                {**base_row, "categoria": None, "categoria_url": None, "categoria_es_interna": None}
            )

    return rows


def parse_total_paginas(html: str) -> Optional[int]:
    soup = BeautifulSoup(html, "html.parser")
    max_page = None
    for a in soup.select("ul.zurb-pagination li a[href]"):
        href = a["href"]
        if "/resultados/" in href:
            tail = href.split("/resultados/")[-1].split("?")[0]
            if tail.isdigit():
                n = int(tail)
                if max_page is None or n > max_page:
                    max_page = n
    return max_page


def fetch_page(session: requests.Session, page: int, max_retries: int = 3, timeout: int = 30) -> Optional[str]:
    url = FIRST_PAGE_URL if page == 1 else LIST_URL_TEMPLATE.format(page=page)
    for attempt in range(1, max_retries + 1):
        try:
            resp = session.get(url, headers=HEADERS, timeout=timeout)
            if resp.status_code == 200:
                return resp.text
            print(f"  [pág {page}] HTTP {resp.status_code} (intento {attempt}/{max_retries})")
        except Exception as e:
            # Cualquier fallo de red/DNS/SSL/lo-que-sea cuenta como reintentable,
            # nunca debe tirar abajo todo el crawler.
            print(f"  [pág {page}] error de red: {e} (intento {attempt}/{max_retries})")
        time.sleep(2 ** attempt)
    return None


def crawl_all(
    out_dir: str = "ccnorte_data",
    delay_seconds: float = 1.5,
    max_pages: Optional[int] = None,
    save_html: bool = True,
):
    """Descarga y parsea todas las páginas del listado de resultados (con checkpoint)."""
    out_path = Path(out_dir)
    html_dir = out_path / "html"
    out_path.mkdir(parents=True, exist_ok=True)
    if save_html:
        html_dir.mkdir(exist_ok=True)

    csv_path = out_path / "ccnorte_resultados.csv"

    done_pages: set[int] = set()
    if csv_path.exists():
        with open(csv_path, encoding="utf-8") as f:
            for row in csv.DictReader(f):
                if row.get("pagina"):
                    done_pages.add(int(row["pagina"]))
        print(f"Checkpoint encontrado: {len(done_pages)} páginas ya procesadas en {csv_path}")

    session = requests.Session()

    total = max_pages
    if total is None:
        html1 = None
        cache1 = html_dir / "pagina_0001.html"
        if save_html and cache1.exists():
            html1 = cache1.read_text(encoding="utf-8")
        if html1 is None:
            print("Descargando página 1 para detectar el total de páginas...")
            html1 = fetch_page(session, 1)
            if html1 is None:
                raise RuntimeError("No se ha podido descargar la página 1")
            if save_html:
                cache1.write_text(html1, encoding="utf-8")
        total = parse_total_paginas(html1)
        if total is None:
            raise RuntimeError("No se ha podido detectar el total de páginas del paginador")
    print(f"Total de páginas a procesar: {total}")

    write_header = not csv_path.exists()
    n_rows_written = 0
    n_pages_ok = 0
    failed_pages: list[int] = []

    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        if write_header:
            writer.writeheader()

        for page in range(1, total + 1):
            if page in done_pages:
                continue

            # Cualquier error inesperado en esta página (parseo raro, disco,
            # lo que sea) se registra y se pasa a la siguiente — nunca debe
            # cortar el resto del crawling.
            try:
                html = None
                cache_file = html_dir / f"pagina_{page:04d}.html"
                if save_html and cache_file.exists():
                    html = cache_file.read_text(encoding="utf-8")
                else:
                    html = fetch_page(session, page)
                    if html is None:
                        failed_pages.append(page)
                        print(f"  [pág {page}] FALLIDA definitivamente, se continúa")
                        continue
                    if save_html:
                        cache_file.write_text(html, encoding="utf-8")
                    time.sleep(delay_seconds)

                rows = parse_resultados_page(html)
                for r in rows:
                    writer.writerow({"pagina": page, **r})
                f.flush()

                n_rows_written += len(rows)
                n_pages_ok += 1
                if page % 10 == 0 or page == total:
                    print(f"  Progreso: página {page}/{total} ({n_rows_written} filas nuevas)")
            except Exception as e:
                failed_pages.append(page)
                print(f"  [pág {page}] ERROR inesperado ({e}), se continúa con la siguiente")
                continue

    print()
    print("=== RESUMEN FASE 1 ===")
    print(f"Páginas procesadas en esta ejecución: {n_pages_ok}")
    if failed_pages:
        print(f"Páginas fallidas (vuelve a ejecutar para reintentarlas): {failed_pages}")
    df = pd.read_csv(csv_path, dtype=str)
    print(f"Total filas en el CSV: {len(df)}")
    print(f"Total carreras únicas: {df['nombre_carrera'].nunique()}")
    print(f"CSV consolidado: {csv_path.resolve()}")
    return df

### Fase 2 — Clasificaciones (`crawl_clasificaciones`)

Cuenta participantes por género con el método del máximo por prefijo de código de posición (`ABF-27`, `ABM-59`...), inmune a páginas repetidas por bucles del paginador. `fecha` y `circuito_nombre` se arrastran desde la Fase 1 hasta el resultado final.

In [ ]:
import hashlib
import re
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import StringIO
from urllib.parse import urlparse, urlencode, parse_qs, urlunparse

MAIN_HOST = "ccnorte.com"
RE_COMPETICION = re.compile(r"/web/resultado/competicion-\d+")
RE_CODIGO = re.compile(r"^(.*?)[\s]*-[\s]*(\d+)$")
RE_GENERO_CATEGORIA = re.compile(r"\b(masculin[oa]|femenin[oa]|mixt[oa]s?)\b", re.IGNORECASE)
RE_GENERO_MF = re.compile(r"\b(masculin[oa]|femenin[oa])\b", re.IGNORECASE)

CAMPOS_PARTICIPANTES = [
    "nombre_carrera", "fecha", "categoria", "url_base", "paginas_leidas",
    "n_masculino", "n_femenino", "n_desconocido", "n_total",
    "n_filas_unicas", "n_filas_sin_codigo",
]

_MESES = {
    "ene": "01", "feb": "02", "mar": "03", "abr": "04", "may": "05", "jun": "06",
    "jul": "07", "ago": "08", "sep": "09", "oct": "10", "nov": "11", "dic": "12",
}

_thread_local = threading.local()


def _session() -> requests.Session:
    if not hasattr(_thread_local, "session"):
        _thread_local.session = requests.Session()
    return _thread_local.session


class RateLimiter:
    """Limita el número global de peticiones/segundo entre todos los threads."""

    def __init__(self, max_per_second: float):
        self.min_interval = 1.0 / max_per_second if max_per_second > 0 else 0.0
        self._lock = threading.Lock()
        self._next_time = 0.0

    def wait(self):
        if self.min_interval <= 0:
            return
        with self._lock:
            now = time.monotonic()
            wait_until = max(self._next_time, now)
            self._next_time = wait_until + self.min_interval
        delay = wait_until - time.monotonic()
        if delay > 0:
            time.sleep(delay)


def _host(url: str) -> str:
    return urlparse(url).netloc.lower()


def _tipo_url(url: str) -> str:
    h = _host(url)
    if h == MAIN_HOST or h == "www." + MAIN_HOST:
        return "interna" if "/resultados/" in url else "externa"
    if h.endswith("." + MAIN_HOST):
        return "competicion" if RE_COMPETICION.search(url) else "subdominio"
    return "externa"


def _url_pagina(url_base: str, page: int) -> str:
    if _tipo_url(url_base) == "interna":
        return re.sub(r"/\d+(\?.*)?$", f"/{page}", url_base)
    parts = urlparse(url_base)
    q = parse_qs(parts.query)
    q["page"] = [str(page)]
    return urlunparse(parts._replace(query=urlencode(q, doseq=True)))


def _fetch(url: str, limiter: RateLimiter, max_retries: int = 3, timeout: int = 30) -> Optional[str]:
    for attempt in range(1, max_retries + 1):
        limiter.wait()
        try:
            resp = _session().get(url, headers=HEADERS, timeout=timeout)
            if resp.status_code == 200:
                return resp.text
            if resp.status_code == 404:
                return None
            if resp.status_code == 429:
                print(f"  [{url}] HTTP 429, esperando...")
                time.sleep(10 * attempt)
                continue
            print(f"  [{url}] HTTP {resp.status_code} (intento {attempt}/{max_retries})")
        except Exception as e:
            # Cualquier fallo (red, DNS, SSL, timeout...) es reintentable, no
            # debe tirar abajo el hilo ni el resto del scraping.
            print(f"  [{url}] error: {e} (intento {attempt}/{max_retries})")
        time.sleep(2 ** attempt)
    return None


def _extraer_tabla_principal(html: str) -> Optional[pd.DataFrame]:
    try:
        tablas = pd.read_html(StringIO(html))
    except Exception:
        # pd.read_html puede lanzar cosas distintas de ValueError (parsers
        # rotos, encodings raros...) — cualquier fallo aquí significa
        # simplemente "esta página no tiene tabla útil", no es fatal.
        return None
    tablas = [t for t in tablas if len(t) > 0]
    if not tablas:
        return None
    tabla = max(tablas, key=len)
    if isinstance(tabla.columns, pd.MultiIndex):
        tabla.columns = [
            " ".join(str(x) for x in col if str(x) != "nan").strip() for col in tabla.columns
        ]
    tabla.columns = [str(c).strip() for c in tabla.columns]
    return tabla.dropna(axis=1, how="all")


def _paginacion_info(html: str) -> tuple[Optional[int], bool]:
    soup = BeautifulSoup(html, "lxml")
    max_page = None
    has_next = False

    for a in soup.select("ul.zurb-pagination li a[href], ul.pagination li a[href]"):
        href = a.get("href", "")
        n = None
        m = re.search(r"[?&]page=(\d+)", href)
        if m:
            n = int(m.group(1))
        else:
            m = re.search(r"/(\d+)(?:\?.*)?$", href)
            if m:
                n = int(m.group(1))
        if n is not None and (max_page is None or n > max_page):
            max_page = n

    next_li = soup.select_one("ul.pagination li.next")
    if next_li is not None and "disabled" not in next_li.get("class", []) \
            and next_li.select_one("a[href]"):
        has_next = True
    else:
        for li in soup.select("ul.zurb-pagination li.arrow"):
            if "unavailable" not in li.get("class", []) and li.select_one("a[href]"):
                a = li.select_one("a[href]")
                if a.get("href", "#") != "#":
                    has_next = True

    return max_page, has_next


def _hash_tabla(tabla: pd.DataFrame) -> str:
    return hashlib.md5(tabla.to_csv(index=False).encode("utf-8", errors="ignore")).hexdigest()


def _acumular_codigos(tabla: pd.DataFrame, max_por_prefijo: dict, filas_sin_codigo: list):
    col_sexo = next((c for c in tabla.columns if "sexo" in str(c).lower()), None)
    col_cat = next((c for c in tabla.columns if "categor" in str(c).lower()
                    and "puesto" in str(c).lower()), None)

    n_sin = 0
    for _, fila in tabla.iterrows():
        codigo = None
        for col in (col_sexo, col_cat):
            if col is None:
                continue
            m = RE_CODIGO.match(str(fila[col]).strip().upper())
            if m and m.group(1):
                codigo = m
                break
        if codigo is None:
            n_sin += 1
            continue
        prefijo, num = codigo.group(1), int(codigo.group(2))
        if num > max_por_prefijo.get(prefijo, 0):
            max_por_prefijo[prefijo] = num
    filas_sin_codigo.append(n_sin)


def _totales_desde_prefijos(max_por_prefijo: dict) -> dict:
    n_m = n_f = n_desc = 0
    for prefijo, mx in max_por_prefijo.items():
        if prefijo.endswith("F"):
            n_f += mx
        elif prefijo.endswith("M"):
            n_m += mx
        else:
            n_desc += mx
    return {"n_masculino": n_m, "n_femenino": n_f, "n_desconocido": n_desc,
            "n_total": n_m + n_f + n_desc}


def _genero_categoria(categoria) -> str:
    """Detecta si 'categoria' ya viene separada por género en su propio
    nombre ("3500m MASCULINO", "Individual Femenina"...)."""
    if not isinstance(categoria, str):
        return "mixto"
    m = RE_GENERO_CATEGORIA.search(categoria)
    if not m:
        return "mixto"
    palabra = m.group(1).lower()
    if palabra.startswith("masculin"):
        return "masculino"
    if palabra.startswith("femenin"):
        return "femenino"
    return "mixto"


def _categoria_base(categoria):
    """Quita 'masculino'/'femenino' del texto de la categoria, para poder
    fusionar en una sola fila el par "X Masculino" + "X Femenino" (misma
    prueba, solo separada por género) sumando sus recuentos. Si categoria
    ERA solo la palabra de género (sin nada más), queda "General" — así
    "Femenina" y "Masculina" sueltas también se fusionan en una sola fila
    (antes, al quedar vacío, se devolvía el texto original y nunca
    fusionaban)."""
    if not isinstance(categoria, str):
        return categoria
    base = RE_GENERO_MF.sub("", categoria)
    base = re.sub(r"\s{2,}", " ", base).strip(" ,.-")
    return base or "General"


def _parsear_fecha(texto):
    """'05 jul. 2026' (mes abreviado en español) -> fecha real."""
    if not isinstance(texto, str) or not texto.strip():
        return None
    partes = texto.lower().replace(".", "").split()
    if len(partes) != 3:
        return None
    dia, mes, anio = partes
    mes_num = _MESES.get(mes[:3])
    if not mes_num:
        return None
    return f"{anio}-{mes_num}-{dia.zfill(2)}"


def descubrir_competiciones(home_url: str, cache_dir: Path, limiter: RateLimiter) -> list[dict]:
    try:
        safe = re.sub(r"[^a-zA-Z0-9]+", "_", home_url).strip("_")
        cache = cache_dir / f"home_{safe}.html"
        if cache.exists():
            html = cache.read_text(encoding="utf-8")
        else:
            html = _fetch(home_url, limiter)
            if html is None:
                return []
            cache.write_text(html, encoding="utf-8")

        soup = BeautifulSoup(html, "lxml")
        vistos: set[str] = set()
        comps: list[dict] = []
        for a in soup.find_all("a", href=True):
            if RE_COMPETICION.search(a["href"]):
                url_abs = urljoin(home_url, a["href"]).split("?")[0].split("#")[0]
                if url_abs in vistos:
                    continue
                vistos.add(url_abs)
                comps.append({"nombre": " ".join(a.get_text().split()) or None, "url": url_abs})
        return comps
    except Exception as e:
        # Un subdominio con HTML raro o roto no debe frenar el resto.
        print(f"  [{home_url}] ERROR inesperado descubriendo competiciones ({e})")
        return []


def crawl_clasificaciones(
    csv_entrada,
    out_dir,
    max_fuentes: Optional[int] = 5,
    max_workers: int = 6,
    max_requests_per_second: float = 4.0,
    max_paginas_por_clasificacion: int = 200,
):
    """
    Recorre en paralelo todas las clasificaciones y escribe UNA fila por
    clasificación con los recuentos por género (método del máximo por
    prefijo), arrastrando fecha desde ccnorte_resultados.csv.
    Ningún error inesperado en una fuente concreta interrumpe las demás.
    """
    out_path = Path(out_dir)
    html_dir = out_path / "html_clasificaciones"
    html_dir.mkdir(parents=True, exist_ok=True)

    csv_salida = out_path / "ccnorte_participantes.csv"
    csv_comps = out_path / "ccnorte_competiciones.csv"
    done_file = out_path / "ccnorte_done_clasificaciones.txt"

    # Guarda de esquema: si el CSV ya existe con una cabecera distinta a la
    # actual, seguir añadiendo filas las desplazaría de columna (esto ya
    # pasó una vez: filas de una versión sin "fecha" se leyeron desplazadas
    # bajo la cabecera nueva, con la URL apareciendo en "categoria"). Mejor
    # parar con un aviso claro que corromper en silencio.
    if csv_salida.exists():
        cabecera_actual = pd.read_csv(csv_salida, nrows=0).columns.tolist()
        if cabecera_actual != CAMPOS_PARTICIPANTES:
            raise RuntimeError(
                f"{csv_salida} tiene una cabecera distinta a la esperada — seguir "
                f"añadiendo filas la corrompería (desplazamiento de columnas).\n"
                f"Esperada:  {CAMPOS_PARTICIPANTES}\n"
                f"Encontrada: {cabecera_actual}\n"
                f"Borra {csv_salida.name} y {done_file.name} en {out_path} "
                f"(la caché de HTML se conserva, no hace falta volver a descargar "
                f"nada) y vuelve a ejecutar."
            )

    df_in = pd.read_csv(csv_entrada, dtype=str).dropna(subset=["categoria_url"])
    df_in["tipo"] = df_in["categoria_url"].map(_tipo_url)

    internas = (
        df_in[df_in["tipo"] == "interna"][
            ["nombre_carrera", "fecha", "categoria", "categoria_url"]
        ].drop_duplicates(subset=["categoria_url"])
    )
    sub = df_in[df_in["tipo"].isin(["subdominio", "competicion"])].copy()
    sub["host"] = sub["categoria_url"].map(_host)
    subdominios = sub.drop_duplicates(subset=["host"])[
        ["nombre_carrera", "fecha", "categoria_url"]
    ]
    externas_fuera = df_in[df_in["tipo"] == "externa"]["categoria_url"].nunique()

    print(f"Fuentes internas (ccnorte.com/resultados): {len(internas)}")
    print(f"Subdominios de carrera (*.ccnorte.com): {len(subdominios)}")
    print(f"URLs externas de terceros (se ignoran): {externas_fuera}")

    fuentes: list[dict] = [
        {"nombre_carrera": r["nombre_carrera"], "fecha": r["fecha"],
         "categoria": r["categoria"], "url": r["categoria_url"], "tipo": "interna"}
        for _, r in internas.iterrows()
    ] + [
        {"nombre_carrera": r["nombre_carrera"], "fecha": r["fecha"],
         "categoria": None, "url": r["categoria_url"], "tipo": "subdominio"}
        for _, r in subdominios.iterrows()
    ]
    if max_fuentes:
        fuentes = fuentes[:max_fuentes]
        print(f"(limitado a {max_fuentes} fuentes para la prueba)")
    print(f"Workers: {max_workers} | Límite global: {max_requests_per_second} peticiones/s")

    done_clas: set[str] = set()
    if done_file.exists():
        done_clas = set(done_file.read_text(encoding="utf-8").splitlines())
        print(f"Checkpoint: {len(done_clas)} clasificaciones ya procesadas")

    limiter = RateLimiter(max_requests_per_second)
    csv_lock = threading.Lock()
    done_lock = threading.Lock()
    print_lock = threading.Lock()

    progreso = {"clasificaciones": 0, "fuentes": 0}
    sin_tabla: list[str] = []
    comps_registradas: list[dict] = []

    def _cache_name(url: str) -> Path:
        safe = re.sub(r"[^a-zA-Z0-9]+", "_",
                      url.replace("https://", "").replace("http://", "")).strip("_")
        return html_dir / f"{safe[:180]}.html"

    def _crawl_una_clasificacion(done_f, nombre_carrera, fecha, categoria, url_base) -> bool:
        with done_lock:
            if url_base in done_clas:
                return True

        max_por_prefijo: dict[str, int] = {}
        filas_sin_codigo: list[int] = []
        hashes_vistos: set[str] = set()
        n_filas_unicas = 0
        paginas_leidas = 0
        max_pag_visible: Optional[int] = None

        page = 1
        while page <= max_paginas_por_clasificacion:
            # Cualquier error inesperado en una página concreta corta la
            # lectura de ESTA clasificación (se queda con lo ya leído) sin
            # tirar abajo el resto del scraping.
            try:
                url = _url_pagina(url_base, page) if page > 1 or _tipo_url(url_base) == "interna" else url_base

                cache = _cache_name(url)
                if cache.exists():
                    html = cache.read_text(encoding="utf-8")
                else:
                    html = _fetch(url, limiter)
                    if html is not None:
                        cache.write_text(html, encoding="utf-8")

                if html is None:
                    break

                tabla = _extraer_tabla_principal(html)
                if tabla is None or len(tabla) == 0:
                    break

                h = _hash_tabla(tabla)
                if h in hashes_vistos:
                    break
                hashes_vistos.add(h)

                _acumular_codigos(tabla, max_por_prefijo, filas_sin_codigo)
                n_filas_unicas += len(tabla)
                paginas_leidas += 1

                pag_max, has_next = _paginacion_info(html)
                if pag_max is not None:
                    max_pag_visible = max(max_pag_visible or 0, pag_max)

                if max_pag_visible is not None and page >= max_pag_visible:
                    break
                if not has_next:
                    break
                page += 1
            except Exception as e:
                print(f"  [{url_base} pág {page}] ERROR inesperado ({e}), se corta aquí")
                break

        if paginas_leidas == 0:
            return False

        counts = _totales_desde_prefijos(max_por_prefijo)
        fila = {
            "nombre_carrera": nombre_carrera,
            "fecha": fecha,
            "categoria": categoria,
            "url_base": url_base,
            "paginas_leidas": paginas_leidas,
            **counts,
            "n_filas_unicas": n_filas_unicas,
            "n_filas_sin_codigo": sum(filas_sin_codigo),
        }
        with csv_lock:
            pd.DataFrame([fila], columns=CAMPOS_PARTICIPANTES).to_csv(
                csv_salida, mode="a", header=not csv_salida.exists(),
                index=False, encoding="utf-8"
            )
        with done_lock:
            done_f.write(url_base + "\n")
            done_f.flush()
            done_clas.add(url_base)
        progreso["clasificaciones"] += 1
        return True

    def _procesar_fuente(fuente: dict, done_f):
        try:
            if fuente["tipo"] == "interna":
                ok = _crawl_una_clasificacion(done_f, fuente["nombre_carrera"], fuente["fecha"],
                                               fuente["categoria"], fuente["url"])
                if not ok:
                    sin_tabla.append(fuente["url"])
            else:
                comps = descubrir_competiciones(fuente["url"], html_dir, limiter)
                for c in comps:
                    comps_registradas.append(
                        {"nombre_carrera": fuente["nombre_carrera"], "home": fuente["url"],
                         "competicion": c["nombre"], "url": c["url"]}
                    )
                    ok = _crawl_una_clasificacion(done_f, fuente["nombre_carrera"], fuente["fecha"],
                                                   c["nombre"], c["url"])
                    if not ok:
                        sin_tabla.append(c["url"])
                if not comps:
                    sin_tabla.append(fuente["url"] + " (ninguna competición descubierta)")
        except Exception as e:
            # Red de seguridad final: nada de lo que pase aquí debe tirar
            # abajo el ThreadPoolExecutor ni las demás fuentes.
            sin_tabla.append(f"{fuente['url']} (error inesperado: {e})")
        finally:
            progreso["fuentes"] += 1
            n = progreso["fuentes"]
            if n % 50 == 0 or n == len(fuentes):
                with print_lock:
                    print(f"  Progreso: {n}/{len(fuentes)} fuentes | "
                          f"{progreso['clasificaciones']} clasificaciones")

    t0 = time.monotonic()
    with open(done_file, "a", encoding="utf-8") as done_f:
        with ThreadPoolExecutor(max_workers=max_workers) as pool:
            futures = [pool.submit(_procesar_fuente, f, done_f) for f in fuentes]
            for fut in as_completed(futures):
                exc = fut.exception()
                if exc:
                    with print_lock:
                        print(f"  [!] Error en una fuente (no debería llegar aquí): {exc}")

    if comps_registradas:
        with csv_lock:
            pd.DataFrame(comps_registradas).to_csv(
                csv_comps, mode="a", header=not csv_comps.exists(),
                index=False, encoding="utf-8"
            )

    mins = (time.monotonic() - t0) / 60
    print()
    print("=== RESUMEN FASE 2 ===")
    print(f"Tiempo: {mins:.1f} min | Clasificaciones procesadas: {progreso['clasificaciones']}")
    if sin_tabla:
        print(f"Fuentes sin tabla/competiciones ({len(sin_tabla)}), muestra:")
        for u in sin_tabla[:10]:
            print(f"  - {u}")
    if csv_salida.exists():
        df_out = pd.read_csv(csv_salida)
        df_out["diferencia_metodos"] = (
            pd.to_numeric(df_out["n_total"], errors="coerce")
            - pd.to_numeric(df_out["n_filas_unicas"], errors="coerce")
        )
        incoherentes = (df_out["diferencia_metodos"].abs()
                        > 0.1 * pd.to_numeric(df_out["n_total"], errors="coerce").clip(lower=1)).sum()
        print(f"Filas en el CSV (1 por clasificación): {len(df_out)}")
        if incoherentes:
            print(f"[AVISO] {incoherentes} clasificaciones con >10% de diferencia entre "
                  f"método del máximo y recuento de filas (revisar columna 'diferencia_metodos')")
        csv_resumen = out_path / "ccnorte_participantes_resumen.csv"
        resumen = resumen_participantes(df_out)
        resumen.to_csv(csv_resumen, index=False, encoding="utf-8")
        print(f"CSV por clasificación: {csv_salida.resolve()}")
        print(f"CSV agregado por carrera+fecha+categoria: {csv_resumen.resolve()}")
        return resumen
    print("No se ha generado ninguna fila.")
    return pd.DataFrame()


def resumen_participantes(df: pd.DataFrame) -> pd.DataFrame:
    """Una fila por (nombre_carrera, fecha, categoria) con totales por género.
    circuito_nombre se descarta a propósito (no interesa para el análisis).
    'fecha' se convierte a fecha real (venía como texto "05 jul. 2026").

    Antes de agrupar, la categoria se reduce a su 'base' quitando la palabra
    de género ("3500m Masculino" / "3500m Femenina" -> "3500m") para que el
    par masculino+femenino de la MISMA prueba se fusione en una sola fila,
    sumando los recuentos de cada uno.

    'categoria_genero' (masculino/femenino/mixto) indica si, tras la fusión,
    la fila sigue siendo de un solo género (no había par) o ya es mixta."""
    df = df.copy()
    for c in ["n_masculino", "n_femenino", "n_desconocido", "n_total"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

    if "fecha" in df.columns:
        df["fecha"] = pd.to_datetime(df["fecha"].apply(_parsear_fecha), errors="coerce")

    df["categoria"] = df["categoria"].apply(_categoria_base)

    claves = ["nombre_carrera"]
    if "fecha" in df.columns:
        claves.append("fecha")
    claves.append("categoria")

    resumen = (
        df.groupby(claves, dropna=False)[
            ["n_masculino", "n_femenino", "n_desconocido", "n_total"]
        ]
        .sum()
        .reset_index()
        .sort_values(["nombre_carrera", "categoria"])
        .reset_index(drop=True)
    )
    resumen["categoria_genero"] = resumen["categoria"].apply(_genero_categoria)
    return resumen

#### Diagnóstico — por qué una clasificación concreta sale con género mal contado

He revisado un HTML real en caché (Samurai Xtreme Race, columna "P. SEXO", códigos "M-1"/"F-1") y para ese caso la lógica funciona bien. Como el sitio usa plantillas de tabla distintas según la carrera (sobre todo en subdominios de terceros), necesito ver un caso real que SÍ falle. Busca en `resumen` una fila con `n_femenino=0` y `n_masculino` bajo, copia su `nombre_carrera`/`categoria`, búscala en `ccnorte_participantes.csv` para sacar la `url_base`, y pásasela a `diagnosticar_conteo()` de abajo.

In [84]:
def diagnosticar_conteo(url_base: str, out_dir=r"../../data/raw/ccnorte", pagina: int = 1):
    """Depuración: muestra qué columna de género detecta, sus valores reales
    y cómo se están contando, para UNA página de una clasificación."""
    out_path = Path(out_dir)
    html_dir = out_path / "html_clasificaciones"

    def _cache_name(url: str) -> Path:
        safe = re.sub(r"[^a-zA-Z0-9]+", "_",
                      url.replace("https://", "").replace("http://", "")).strip("_")
        return html_dir / f"{safe[:180]}.html"

    url = _url_pagina(url_base, pagina) if pagina > 1 or _tipo_url(url_base) == "interna" else url_base
    cache = _cache_name(url)
    if not cache.exists():
        print(f"No tengo esta página en caché: {cache}")
        print("(prueba con pagina=1, o comprueba que la url_base es exacta)")
        return

    html = cache.read_text(encoding="utf-8")
    tabla = _extraer_tabla_principal(html)
    if tabla is None:
        print("No se ha encontrado ninguna tabla en esta página.")
        return

    print("Columnas detectadas en la tabla:", list(tabla.columns))

    col_sexo = next((c for c in tabla.columns if "sexo" in str(c).lower()), None)
    col_cat = next((c for c in tabla.columns if "categor" in str(c).lower()
                    and "puesto" in str(c).lower()), None)
    print(f"col_sexo detectada: {col_sexo!r}")
    print(f"col_cat detectada:  {col_cat!r}  (necesita 'categor' Y 'puesto' en el nombre)")
    print()

    if col_sexo is not None:
        print(f"Primeros 15 valores de {col_sexo!r}:")
        print(tabla[col_sexo].head(15).tolist())
    if col_cat is not None:
        print(f"Primeros 15 valores de {col_cat!r}:")
        print(tabla[col_cat].head(15).tolist())
    if col_sexo is None and col_cat is None:
        print("¡Ninguna columna reconocida como sexo ni como categoría con "
              "'puesto'! Estas son TODAS las columnas por si alguna es la "
              "correcta con otro nombre:")
        for c in tabla.columns:
            print(f"  {c!r} -> {tabla[c].head(3).tolist()}")

    print()
    max_por_prefijo = {}
    filas_sin_codigo = []
    _acumular_codigos(tabla, max_por_prefijo, filas_sin_codigo)
    print("max_por_prefijo:", max_por_prefijo)
    print("filas sin código reconocible:", filas_sin_codigo)
    print("totales:", _totales_desde_prefijos(max_por_prefijo))

### Orquestador — una sola llamada para las dos fases

In [85]:
def scrape_ccnorte_completo(
    out_dir: str = r"../../data/raw/ccnorte",
    delay_seconds_calendario: float = 1.5,
    max_fuentes_clasificaciones: Optional[int] = None,
    max_workers: int = 6,
    max_requests_per_second: float = 4.0,
):
    """
    Ejecuta las dos fases una detrás de otra:
      1. crawl_all()            -> ccnorte_resultados.csv (calendario)
      2. crawl_clasificaciones() -> ccnorte_participantes_resumen.csv
         (ya con fecha/circuito_nombre incluidos, sin que haga falta ningún join)

    Se puede interrumpir y volver a llamar: las dos fases tienen su propio
    checkpoint (páginas de calendario / clasificaciones ya vistas).
    """
    print("=== FASE 1: calendario (ccnorte_resultados.csv) ===")
    crawl_all(out_dir=out_dir, delay_seconds=delay_seconds_calendario)

    print()
    print("=== FASE 2: clasificaciones (ccnorte_participantes_resumen.csv) ===")
    csv_entrada = str(Path(out_dir) / "ccnorte_resultados.csv")
    resumen = crawl_clasificaciones(
        csv_entrada=csv_entrada,
        out_dir=out_dir,
        max_fuentes=max_fuentes_clasificaciones,
        max_workers=max_workers,
        max_requests_per_second=max_requests_per_second,
    )

    print()
    print("=== LISTO ===")
    print(f"Tabla final (con fecha y circuito_nombre ya incluidos, sin join): "
          f"{Path(out_dir) / 'ccnorte_participantes_resumen.csv'}")
    return resumen

### Uso

Primero una prueba pequeña (`max_fuentes_clasificaciones=20`) para comprobar que todo funciona, y luego sin límite para procesarlo entero. Se puede interrumpir y volver a ejecutar la misma celda: sigue por el checkpoint.

In [86]:
# Limpieza única: ccnorte_participantes.csv quedó corrompido mezclando una
# versión antigua (sin "fecha") con la actual — lo borramos junto a su
# checkpoint para regenerarlo limpio. La caché de HTML se conserva (no hay
# que volver a descargar nada, solo re-parsear).
OUT_DIR = Path("../../data/raw/ccnorte")
_out = Path(OUT_DIR)

(_out / "ccnorte_participantes.csv").unlink(missing_ok=True)
(_out / "ccnorte_participantes_resumen.csv").unlink(missing_ok=True)
(_out / "ccnorte_done_clasificaciones.txt").unlink(missing_ok=True)

print("Listo. La próxima vez que ejecutes la Fase 2 se regenerará limpia desde la caché de HTML.")

Listo. La próxima vez que ejecutes la Fase 2 se regenerará limpia desde la caché de HTML.


In [87]:
# Para que se vea la tabla entera sin truncar columnas ni texto:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

# Prueba pequeña primero:
resumen = scrape_ccnorte_completo(max_fuentes_clasificaciones=None)
resumen.head(20)

=== FASE 1: calendario (ccnorte_resultados.csv) ===
Checkpoint encontrado: 364 páginas ya procesadas en C:\Users\Clàudia Rafart\OneDrive - Prenomics\Escritorio\Altres\ccnorte_data\ccnorte_resultados.csv
Total de páginas a procesar: 364

=== RESUMEN FASE 1 ===
Páginas procesadas en esta ejecución: 0
Total filas en el CSV: 9340
Total carreras únicas: 3609
CSV consolidado: C:\Users\Clàudia Rafart\OneDrive - Prenomics\Escritorio\Altres\ccnorte_data\ccnorte_resultados.csv

=== FASE 2: clasificaciones (ccnorte_participantes_resumen.csv) ===
Fuentes internas (ccnorte.com/resultados): 6782
Subdominios de carrera (*.ccnorte.com): 39
URLs externas de terceros (se ignoran): 866
Workers: 6 | Límite global: 4.0 peticiones/s
  Progreso: 50/6821 fuentes | 50 clasificaciones
  Progreso: 100/6821 fuentes | 100 clasificaciones
  Progreso: 150/6821 fuentes | 149 clasificaciones
  Progreso: 200/6821 fuentes | 199 clasificaciones
  Progreso: 250/6821 fuentes | 249 clasificaciones
  Progreso: 300/6821 fuent

,nombre_carrera,fecha,categoria,n_masculino,n_femenino,n_desconocido,n_total,categoria_genero
0,"""200 CRESTAS"" DESAFÍO COUREL 2016",2016-03-19,92 KM (Desnivel positivo: 3.702 metros - Desnivel negativo: 3.704 metros),201,2,0,203,mixto
1,"""I TRAIL 175º ANIVERSARIO GUARDIA CIVIL"" A BENEFICIO DE LA FUNDACIÓN PADRE RUBINOS",2019-05-12,TRAIL,159,33,0,192,mixto
2,#MarzoSaudeSolidario 2015,2015-05-10,CARRERA CATEGORIA A (2006-2008),8,4,0,12,mixto
3,#MarzoSaudeSolidario 2015,2015-05-10,CARRERA CATEGORIA B (2003-2005),4,10,0,14,mixto
4,#MarzoSaudeSolidario 2015,2015-05-10,CARRERA CATEGORIA C (2000-2002),7,5,0,12,mixto
5,#MarzoSaudeSolidario 2015,2015-05-10,CARRERA POPULAR,49,22,0,71,mixto
6,#MarzoSaudeSolidario 2015,2015-05-10,TRAIL,37,6,0,43,mixto
7,'CLÁSICA' DESAFÍO COUREL 2016,2016-09-24,98 KM (Desnivel positivo: 4.101 metros - Desnivel negativo: 4.103 metros),115,4,0,119,mixto
8,+10 MARIN MANOLO ROSALES. GRAN PREMIO ENCE 2012,2012-08-03,Adultos. Distancia: 10000 m,0,0,0,0,mixto
9,+10MARIN MANUEL ROSALES - GRAN PREMIO ENCE 2014,2014-08-01,PROBA ABSOLUTA. DISTANCIA: 10000 M,0,0,0,0,mixto


### Fase 3 (opcional) — Geocodificación del lugar de la carrera

ccnorte.com **no** da ninguna ubicación geográfica (comprobado en el HTML: ni en el calendario ni en las clasificaciones). Como alternativa, se intenta **extraer el nombre del lugar del propio `nombre_carrera`** (heurística: quita numerales romanos/ordinales, año, texto entre paréntesis y palabras genéricas de tipo de carrera — "CARREIRA POPULAR DE LEIRO" -> "LEIRO") y se geocodifica con **Nominatim/OpenStreetMap** (gratis, sin API key).

**Aviso — esto es un best-effort, no siempre acertará:** nombres de patrocinadores, memoriales con nombre de persona, títulos sin ningún lugar reconocible, etc. pueden dar una ubicación incorrecta o ninguna. Revisa `ccnorte_ubicaciones.csv` (columna `candidato_lugar`) antes de confiar en `ubicacion`/`lat`/`lon`.

Requiere `pip install geopy`. Nominatim limita a 1 petición/segundo — para ~3500-6700 carreras únicas puede tardar 1-2 horas; tiene checkpoint propio (`ccnorte_ubicaciones.csv`), se puede interrumpir y continuar.

In [88]:
_RE_NUMERAL_INICIAL = re.compile(r"^\s*(?:[ivxlcdm]+|\d+)[ºªo\.]*\s+", re.IGNORECASE)
_RE_ANIO = re.compile(r"\b(19|20)\d{2}\b")

_PALABRAS_GENERICAS = {
    "carreira", "carrera", "popular", "trail", "cross", "maraton", "maratón",
    "medio", "travesia", "travesía", "memorial", "trofeo", "circuito",
    "campeonato", "campionato", "duatlon", "duatlón", "triatlon", "triatlón",
    "nocturna", "nocturno", "solidaria", "solidario", "btt", "ruta",
    "andaina", "andaina/carreira", "gran", "premio", "edicion", "edición",
    "subida", "volta", "copa", "liga", "superliga", "xogade",
    "internacional", "provincial", "concello", "ayuntamiento", "concello de",
    "villa", "vila", "ciudad", "cidade", "nado", "milla", "vuelta",
    "gladiator", "warrior", "race", "run", "running", "extreme", "night",
    "infantil", "escolar", "cto", "xunta", "galicia", "final", "comarcal",
    "fase", "previa", "cta",
}


def _candidato_lugar(nombre_carrera: str) -> str:
    """Heurística best-effort para aislar el nombre de lugar dentro del
    título de la carrera. No es NLP de verdad, solo reglas simples — falla
    con nombres de patrocinadores, memoriales con nombre de persona, etc."""
    if not isinstance(nombre_carrera, str) or not nombre_carrera.strip():
        return ""

    texto = nombre_carrera.strip()
    texto = _RE_NUMERAL_INICIAL.sub("", texto)

    # El lugar suele venir DESPUÉS del último "de"/"do"/"da"/"en"/"d'"
    partes = re.split(r"\b(?:de|do|da|en|d')\b", texto, flags=re.IGNORECASE)
    candidato = partes[-1].strip() if len(partes) > 1 else texto

    candidato = _RE_ANIO.split(candidato)[0]
    candidato = candidato.split("(")[0]

    palabras = [
        p for p in candidato.split()
        if p.lower().strip(",.-") not in _PALABRAS_GENERICAS
    ]
    candidato = " ".join(palabras).strip(" ,.-")

    return candidato or texto


def geocodificar_ubicaciones(
    nombres_carrera,
    out_dir,
    pausa_segundos: float = 1.1,
):
    """
    Geocodifica (best-effort) el lugar de cada carrera a partir de su
    nombre, usando Nominatim/OpenStreetMap (gratis, sin API key, límite de
    1 petición/segundo). Cachea en ccnorte_ubicaciones.csv (checkpoint) para
    no repetir geocodificaciones ya hechas si se interrumpe.
    """
    from geopy.geocoders import Nominatim
    from geopy.exc import GeopyError

    out_path = Path(out_dir)
    csv_ubic = out_path / "ccnorte_ubicaciones.csv"

    cache: dict[str, dict] = {}
    if csv_ubic.exists():
        prev = pd.read_csv(csv_ubic, dtype=str)
        cache = {row["nombre_carrera"]: row.to_dict() for _, row in prev.iterrows()}
        print(f"Checkpoint: {len(cache)} carreras ya geocodificadas")

    geolocator = Nominatim(user_agent="ccnorte_scraper_claudia")

    nombres_unicos = list(dict.fromkeys(n for n in nombres_carrera if isinstance(n, str)))
    pendientes = [n for n in nombres_unicos if n not in cache]
    print(f"Carreras a geocodificar: {len(pendientes)} (de {len(nombres_unicos)} únicas)")

    write_header = not csv_ubic.exists()
    with open(csv_ubic, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f, fieldnames=["nombre_carrera", "candidato_lugar", "ubicacion", "lat", "lon"]
        )
        if write_header:
            writer.writeheader()

        for i, nombre in enumerate(pendientes, 1):
            candidato = _candidato_lugar(nombre)
            fila = {"nombre_carrera": nombre, "candidato_lugar": candidato,
                    "ubicacion": None, "lat": None, "lon": None}
            try:
                loc = geolocator.geocode(
                    f"{candidato}, España", exactly_one=True, country_codes="es", timeout=10,
                )
                if loc:
                    fila["ubicacion"] = loc.address
                    fila["lat"] = loc.latitude
                    fila["lon"] = loc.longitude
            except GeopyError as e:
                print(f"  [{nombre}] error de geocodificación: {e}")
            except Exception as e:
                print(f"  [{nombre}] ERROR inesperado: {e}")

            writer.writerow(fila)
            f.flush()
            cache[nombre] = fila

            if i % 50 == 0 or i == len(pendientes):
                print(f"  Progreso: {i}/{len(pendientes)}")
            time.sleep(pausa_segundos)  # respeta el límite de Nominatim (1 req/s)

    print(f"CSV de ubicaciones: {csv_ubic.resolve()}")
    df_ubic = pd.DataFrame(cache.values())
    print(f"Geocodificadas con éxito: {df_ubic['ubicacion'].notna().sum()} de {len(df_ubic)}")
    return df_ubic

In [89]:
OUT_DIR = Path("../../data/raw/ccnorte")

# Prueba pequeña primero: geocodifica solo las carreras que ya tienes en `resumen`.
df_ubicaciones = geocodificar_ubicaciones(resumen["nombre_carrera"], out_dir=OUT_DIR)

resumen_con_ubicacion = resumen.merge(
    df_ubicaciones[["nombre_carrera", "candidato_lugar", "ubicacion", "lat", "lon"]],
    on="nombre_carrera", how="left",
)
resumen_con_ubicacion.sample(20)

Checkpoint: 2479 carreras ya geocodificadas
Carreras a geocodificar: 0 (de 2479 únicas)
CSV de ubicaciones: C:\Users\Clàudia Rafart\OneDrive - Prenomics\Escritorio\Altres\ccnorte_data\ccnorte_ubicaciones.csv
Geocodificadas con éxito: 1365 de 2479


,nombre_carrera,fecha,categoria,n_masculino,n_femenino,n_desconocido,n_total,categoria_genero,candidato_lugar,ubicacion,lat,lon
6255,XXXVII CARREIRA POPULAR CONCELLO DE MEAÑO,2023-09-10,CIRCUITO MEDIANO (2010-2015),56,43,0,99,mixto,MEAÑO,"Meaño, O Salnés, Pontevedra, Galicia, España",42.4529577,-8.7915587
5880,XXIX VOLTA PEDESTRE A AIOS,2016-04-24,INFANTÍS (2004-2003) E CADETES (2002-2001),4,5,0,9,mixto,PEDESTRE A AIOS,NaN,NaN,NaN
573,CARREIRA DO CENTRO (CxOU 2018),2018-06-24,CARREIRA ABSOLUTA (2002 E ANOS ANTERIORES),267,48,0,315,mixto,CENTRO,"Centro, Madrid, Comunidad de Madrid, España",40.4176527,-3.7079547
830,CTO. GALEGO DE AUGAS ABERTAS - PONTEAREAS 2022,2022-06-04,1250M ALEVIN,12,16,0,28,mixto,AUGAS ABERTAS - PONTEAREAS,NaN,NaN,NaN
1794,II TRAVESIA A NADO RIO LEREZ,2024-07-27,ELITE (2010 e anteriores),18,9,0,27,mixto,A RIO LEREZ,"O Río Lérez, Posmarcos, A Pobra do Caramiñal, A Barbanza, A Coruña, Galicia, 15948, España",42.6156519,-8.9335932
3207,SAN SILVESTRE RIBADEO 2016,2016-12-31,MILLA (2005 AL 2008),36,21,0,57,mixto,SAN SILVESTRE RIBADEO,NaN,NaN,NaN
1698,II EDICION 24 HORAS DE VIGO,2015-08-29,INDIVIDUAL,0,0,0,0,mixto,VIGO,"Vigo, Pontevedra, Galicia, España",42.1986395,-8.727956
5773,XXIII COPA ASTURIAS DE NATACION DE AGUAS ABIERTAS,2022-08-13,Individual (2008 y anteriores),91,32,0,123,mixto,AGUAS ABIERTAS,NaN,NaN,NaN
5357,"XVI CARREIRA PEDESTRE CONCELLO DE MOAÑA ""HOMOLOGADA 10K""",2019-05-05,INFANTIL (2006-2007),6,7,0,13,mixto,"MOAÑA ""HOMOLOGADA 10K""",NaN,NaN,NaN
407,8KM CASTRILLÓN 2019 (ASTURIAS),2019-12-21,BENJAMIN (7 a 8 años),52,66,0,118,mixto,8KM CASTRILLÓN,"Castrillón, Asturias / Asturies, España",43.5458244,-5.9924295
